# Olostep Spark Integration Demo

Enrich company data at scale using Olostep directly inside PySpark SQL queries.

Inspired by [parallel-web-tools](https://docs.parallel.ai/data-integrations/spark) — same UDF pattern, powered by Olostep.

## Installation

In [ ]:
# pip install olostep-spark[spark]

## Setup: Initialize SparkSession and Register Olostep UDFs

In [ ]:
import os
from pyspark.sql import SparkSession
from olostep_spark import register_olostep_udfs

spark = SparkSession.builder.appName("olostep-demo").getOrCreate()

register_olostep_udfs(
    spark,
    api_key=os.environ["OLOSTEP_API_KEY"],
)
print("Olostep UDFs registered: olostep_enrich, olostep_search")

## Sample Data: Create Company Dataset

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW companies AS
    SELECT 'Google' as company_name, 'https://google.com' as website
    UNION ALL SELECT 'Anthropic', 'https://anthropic.com'
    UNION ALL SELECT 'OpenAI', 'https://openai.com'
    UNION ALL SELECT 'Stripe', 'https://stripe.com'
""")

spark.sql("SELECT * FROM companies").show()

## Enrichment: Basic Data Enrichment with olostep_enrich

In [ ]:
result = spark.sql("""
    SELECT
        company_name,
        olostep_enrich(
            map('company_name', company_name, 'website', website),
            array('CEO name', 'founding year', 'company description')
        ) as enriched_data
    FROM companies
""")

result.show(truncate=False)

## Parsing: Extract Individual Fields from JSON Results

In [ ]:
from pyspark.sql.functions import get_json_object, col

parsed = spark.sql("""
    SELECT
        company_name,
        get_json_object(enriched_data, '$.ceo_name') as ceo,
        get_json_object(enriched_data, '$.founding_year') as founded,
        get_json_object(enriched_data, '$.company_description') as description
    FROM (
        SELECT
            company_name,
            olostep_enrich(
                map('company_name', company_name),
                array('CEO name', 'founding year', 'company description')
            ) as enriched_data
        FROM companies
    )
""")

parsed.show(truncate=80)

## Web Search: Query the Web with olostep_search

In [ ]:
result = spark.sql("""
    SELECT
        company_name,
        olostep_search(company_name || ' latest news') as search_results
    FROM companies
    LIMIT 2
""")

import json
for row in result.collect():
    links = json.loads(row["search_results"])
    if isinstance(links, list) and links:
        print(f"{row['company_name']}: {links[0]['title']} — {links[0]['url']}")

## Best Practices

### Partition Sizing
Use `repartition(n)` where `n ≈ total_rows / 20` for balanced concurrent processing:
```python
df.repartition(df.count() // 20).createOrReplaceTempView("my_table")
```

### Error Handling
Failed rows return `{"error": "..."}` — filter with `get_json_object(col, '$.error').isNull()`:
```python
clean = result.filter(get_json_object(col("enriched"), "$.error").isNull())
```

### Cost Optimization
Each row = one Olostep API call — use `LIMIT` while testing

### Field Naming
Output columns become snake_case keys: `'CEO name'` → `ceo_name`